# QTagger+ — Final QSVM v3.1 Training & Inference

**CIC-MalMem-2022 → original v3 preprocessing → corrected QSVM v3.1 → joblib model → reloaded predictions.**

This is the implementation notebook for the [QIntern26 repository](https://github.com/shanmukhdatta/Qintern26). It follows:
- \`Week_1/code/MalMem_Training.ipynb\` for dataset inspection, classical baselines and evaluation;
- \`Week_6/Day_5a/day5a_final/pipeline_v3_1_fixed.py\` for **corrected** quantum circuit, XGBoost selection, local kernel, KTA and Nyström;
- \`Week_5/Day34_V3_Supplementary/model/load_and_predict_example.py\` for the idea of saving all inference components, **not** its outdated pre-correction model.

**Defaults:** original real CSV, **three malware families** (Ransomware, Spyware, Trojan), 8 qubits, 300 QSVM training samples, 120 test samples, 40 Nyström landmarks. Set \`TASK="four_class"\` to include Benign (a new experiment; previous reported accuracies do not apply).

**Scientific safeguards:** train/test split before fitting scalers or selectors; remove exact duplicate feature rows before split; same train/test rows for quantum and classical comparisons; no CTGAN/SMOTE data in the primary model; corrected v3.1 trainable circuit. The local overlap measurement is not assumed to be a mathematically PSD kernel; Nyström landmark matrix is symmetrized and eigenvalues regularized, and diagnostic results are printed.

**Requirements:** Python 3.10–3.12 recommended, internet access for one-time dataset download, PennyLane, PennyLane-Lightning (optional accelerator), scikit-learn, XGBoost, pandas, NumPy, SciPy, matplotlib, joblib. Training is computationally expensive; use CPU runtime and start with the default sample budget.

**Important:** This notebook is supplied as source code, not as a claimed trained or verified model. Run every cell and inspect its outputs before using the resulting artifacts.

## 0 — Install dependencies (run once if necessary)

In Jupyter/Colab, uncomment and execute the next cell. After installation, restart the runtime if prompted. Keep the generated \`environment.json\` alongside the saved model.

In [ ]:
# %pip install "numpy>=1.26,<3" "pandas>=2.1,<3" "scipy>=1.11,<2" "scikit-learn>=1.4,<2" "xgboost>=2,<4" "pennylane>=0.40,<1" "pennylane-lightning>=0.40,<1" "matplotlib>=3.8,<4" "joblib>=1.3,<2"

## 1 — Imports and reproducible configuration

In [ ]:
import json, os, sys, time, hashlib, platform, warnings
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import scipy
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
import xgboost
import sklearn
import pennylane as qml

SEED = 7
TASK = "malware3"     # "malware3" or "four_class"
N_QUBITS = 8
LAYERS = 2
TRAIN_SUBSET = 300
TEST_SUBSET = 120      # historical v3 runs used only 40; 120 is a stronger demo check
ALIGN_SUBSET = 10
ALIGN_MAXITER = 12
N_LANDMARKS = 40
SVM_C = 1.0
TEST_FRACTION = 0.30
DROP_EXACT_DUPLICATES = True

DATA_PATH = Path("Datasets_/MalMem2022.csv")
OUTPUT_DIR = Path("qtagger_outputs") / TASK
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)

print("Task:", TASK, "| seed:", SEED, "| qubits:", N_QUBITS)
print("Versions:", {"python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__, "sklearn": sklearn.__version__,
    "xgboost": xgboost.__version__, "pennylane": qml.__version__})

## 2 — Load the **original real** MalMem CSV (not the synthetic CSV)

In [ ]:
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/shanmukhdatta/Qintern26/main/Datasets_/MalMem2022.csv"
    print("Downloading original dataset from your GitHub repository ...")
    urlretrieve(url, DATA_PATH)
assert DATA_PATH.exists(), "Dataset not found. Place MalMem2022.csv at DATA_PATH."
df = pd.read_csv(DATA_PATH, low_memory=False)
assert "Category" in df.columns, "Expected the original MalMem2022.csv with Category column."
sha = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("CSV:", DATA_PATH, "| shape:", df.shape, "| SHA256:", sha)
print("Category examples:", df["Category"].value_counts().head(12).to_dict())
display(df.head(3))

## 3 — Define labels, numerical features, and duplicate audit

The corrected v3.1 original-data loader excludes Benign and derives the coarse family from the \`Category\` prefix. We preserve that for \`malware3\`. The optional four-class task adds Benign. Non-feature identifiers (\`Class\`, \`Category\`, \`Filename\`) are never model inputs. Exact duplicate numerical feature rows are removed *before* splitting to prevent identical feature vectors appearing in both partitions.

In [ ]:
category = df["Category"].astype(str).str.strip()
family = category.str.split("-", n=1).str[0]
if TASK == "malware3":
    valid = family.isin(["Ransomware", "Spyware", "Trojan"])
    class_order = ["Ransomware", "Spyware", "Trojan"]
elif TASK == "four_class":
    valid = family.isin(["Benign", "Ransomware", "Spyware", "Trojan"])
    class_order = ["Benign", "Ransomware", "Spyware", "Trojan"]
else:
    raise ValueError("TASK must be malware3 or four_class")

working = df.loc[valid].copy().reset_index(drop=True)
labels = family.loc[valid].reset_index(drop=True)
excluded = {"Class", "Category", "Filename", "Family", "fam"}
raw_feature_cols = [c for c in working.select_dtypes(include=[np.number]).columns if c not in excluded]
assert len(raw_feature_cols) >= N_QUBITS, "Too few numeric features."
X_raw = working[raw_feature_cols].replace([np.inf, -np.inf], np.nan)
assert not X_raw.isna().all(axis=0).any(), "A numeric feature is entirely missing."
print("Before duplicate removal:", len(X_raw), "| feature columns:", len(raw_feature_cols))
dup_mask = X_raw.duplicated(keep="first")
print("Exact duplicate feature rows:", int(dup_mask.sum()))
if DROP_EXACT_DUPLICATES:
    keep = ~dup_mask
    X_raw = X_raw.loc[keep].reset_index(drop=True)
    labels = labels.loc[keep].reset_index(drop=True)
    working = working.loc[keep].reset_index(drop=True)
print("After duplicate audit:", len(X_raw), "| class counts:", labels.value_counts().to_dict())
assert set(labels.unique()) == set(class_order), "One or more expected classes are missing."

## 4 — Freeze the train/test split

Use stratification by default. If \`Filename\` repeats across multiple records, group by filename to prevent one named sample appearing in both sets. The test set is untouched by fitting, feature selection, KTA, or model selection. All comparisons below use **exactly the same sampled test indices**.

In [ ]:
all_idx = np.arange(len(X_raw))
if "Filename" in working.columns and working["Filename"].notna().any() and working["Filename"].duplicated().any():
    groups = working["Filename"].fillna("__missing__").astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=SEED)
    tr_idx, te_idx = next(splitter.split(X_raw, labels, groups=groups))
    split_method = "GroupShuffleSplit by Filename"
else:
    tr_idx, te_idx = train_test_split(all_idx, test_size=TEST_FRACTION,
                                    random_state=SEED, stratify=labels)
    split_method = "Stratified train_test_split"
X_train_raw, X_test_raw = X_raw.iloc[tr_idx].copy(), X_raw.iloc[te_idx].copy()
y_train_text = labels.iloc[tr_idx].to_numpy()
y_test_text = labels.iloc[te_idx].to_numpy()
assert set(class_order).issubset(set(y_train_text))
assert set(class_order).issubset(set(y_test_text))
assert not set(tr_idx).intersection(set(te_idx))
np.savez(OUTPUT_DIR / "frozen_split_indices.npz", train=tr_idx, test=te_idx)
le = LabelEncoder().fit(class_order)
y_train = le.transform(y_train_text)
y_test = le.transform(y_test_text)
print("Split:", split_method, "| train:", len(tr_idx), "| test:", len(te_idx))
print("Train distribution:", pd.Series(y_train_text).value_counts().to_dict())
print("Test distribution:", pd.Series(y_test_text).value_counts().to_dict())

## 5 — Existing v3 preprocessing: variance → correlation → signed-log → StandardScaler

The **same transformations** used by the corrected Week 6 pipeline are applied, but all fitted statistics come from the training set. Missing values, if any, use train medians.

In [ ]:
fill_values = X_train_raw.median(numeric_only=True).fillna(0)
Xtr = X_train_raw.fillna(fill_values).copy()
Xte = X_test_raw.fillna(fill_values).copy()
variances = Xtr.var()
keep_variance = variances[variances > 1e-8].index.tolist()
Xtr = Xtr[keep_variance]
Xte = Xte[keep_variance]
corr = Xtr.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if (upper[c] > 0.95).any()]
kept_columns = [c for c in keep_variance if c not in drop_corr]
Xtr = Xtr[kept_columns]
Xte = Xte[kept_columns]
assert len(kept_columns) >= N_QUBITS, "Fewer than 8 features remain."
def signed_log(frame):
    return np.sign(frame) * np.log1p(np.abs(frame))
standard_scaler = StandardScaler().fit(signed_log(Xtr))
Xtr_s = standard_scaler.transform(signed_log(Xtr))
Xte_s = standard_scaler.transform(signed_log(Xte))
assert np.isfinite(Xtr_s).all() and np.isfinite(Xte_s).all()
print("Features:", len(raw_feature_cols), "→ variance:", len(keep_variance),
      "→ correlation:", len(kept_columns))
print("Removed correlated columns:", drop_corr[:15])

## 6 — XGBoost top-8 feature selection and quantum-angle scaling

This is the **v3 XGBoost feature-selection variant**, not the alternative LDA+PCA variant. Fit XGBoost on the training set only. Then fit \`RobustScaler\` on selected training features and map clipped values to **[-π, π]** as the corrected code does.

In [ ]:
selector = XGBClassifier(n_estimators=100, max_depth=4, random_state=SEED,
                         eval_metric="mlogloss", tree_method="hist", n_jobs=4)
selector.fit(Xtr_s, y_train)
selected_idx = np.argsort(selector.feature_importances_)[::-1][:N_QUBITS]
selected_feature_names = [kept_columns[int(i)] for i in selected_idx]
robust_scaler = RobustScaler().fit(Xtr_s[:, selected_idx])
def to_angles(z):
    return np.clip(robust_scaler.transform(z[:, selected_idx]), -3, 3) / 3 * np.pi
Xtr_q_full = to_angles(Xtr_s)
Xte_q_full = to_angles(Xte_s)
print("Top features:", selected_feature_names)
print("Angle range:", Xtr_q_full.min(), Xtr_q_full.max())

## 7 — Select **identical** controlled training/test subsets for QSVM and baselines

Quantum simulation is expensive. The saved test indices are sampled only from the held-out real-data test partition; both quantum and classical models evaluate on these exact rows. The full holdout remains available for later, larger validation. We do not claim that a 120-row subset proves generalization.

In [ ]:
def stratified_subset_indices(y, size, seed):
    if size is None or size >= len(y):
        return np.arange(len(y))
    if size < len(np.unique(y)) * 2:
        raise ValueError("Sample budget too small for stratification.")
    idx, _ = train_test_split(np.arange(len(y)), train_size=size,
                              stratify=y, random_state=seed)
    return np.sort(idx)

train_sel = stratified_subset_indices(y_train, TRAIN_SUBSET, SEED + 101)
test_sel = stratified_subset_indices(y_test, TEST_SUBSET, SEED + 202)
Xq_train, Xq_test = Xtr_q_full[train_sel], Xte_q_full[test_sel]
yq_train, yq_test = y_train[train_sel], y_test[test_sel]
np.savez(OUTPUT_DIR / "quantum_subset_indices.npz", train=train_sel, test=test_sel)
print("Matched experiment:", len(train_sel), "train,", len(test_sel), "test")
print("Test counts:", dict(zip(*np.unique(le.inverse_transform(yq_test), return_counts=True))))

## 8 — Corrected QSVM v3.1 local quantum circuit

The old v3.0 implementation had adjacent \`RY(w)\` and \`RY(-w)\` gates that cancelled. This implementation matches the corrected Week 6 architecture: the trainable rotations are interleaved **inside** the encoding layers, and the reverse operation is the exact adjoint. The similarity is the average of pairwise \`P(00)\` values from the overlap circuit.

In [ ]:
try:
    dev = qml.device("lightning.qubit", wires=N_QUBITS)
    DEVICE_NAME = "lightning.qubit"
except Exception as exc:
    warnings.warn(f"Lightning unavailable ({exc}); using slower default.qubit.")
    dev = qml.device("default.qubit", wires=N_QUBITS)
    DEVICE_NAME = "default.qubit"

pairs = [(i, i + 1) for i in range(0, N_QUBITS - 1, 2)]
state_ids = np.arange(2 ** N_QUBITS)
pair_masks = [
    (((state_ids >> (N_QUBITS - 1 - a)) & 1) == 0) &
    (((state_ids >> (N_QUBITS - 1 - b)) & 1) == 0)
    for a, b in pairs
]

@qml.qnode(dev)
def overlap_probs(x1, x2, w):
    for _ in range(LAYERS):
        qml.AngleEmbedding(x1, wires=range(N_QUBITS), rotation="Y")
        for i in range(N_QUBITS):
            qml.RY(w[i], wires=i)
        for i in range(N_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % N_QUBITS])
    for _ in range(LAYERS):
        for i in reversed(range(N_QUBITS)):
            qml.CNOT(wires=[i, (i + 1) % N_QUBITS])
        for i in range(N_QUBITS):
            qml.RY(-w[i], wires=i)
        qml.adjoint(qml.AngleEmbedding)(x2, wires=range(N_QUBITS), rotation="Y")
    return qml.probs(wires=range(N_QUBITS))

def local_similarity(x1, x2, w):
    probs = np.asarray(overlap_probs(x1, x2, w))
    return float(np.mean([probs[mask].sum() for mask in pair_masks]))

print("Device:", DEVICE_NAME, "| pairs:", pairs)
w_probe = 0.05 * np.random.default_rng(SEED).standard_normal(N_QUBITS)
diag = local_similarity(Xq_train[0], Xq_train[0], w_probe)
print("Self similarity:", diag)
assert np.isclose(diag, 1.0, atol=1e-6), "Overlap circuit failed K(x,x)=1 check."
k12 = local_similarity(Xq_train[0], Xq_train[1], w_probe)
k21 = local_similarity(Xq_train[1], Xq_train[0], w_probe)
print("Symmetry diagnostic:", k12, k21, "| abs difference:", abs(k12-k21))
print("Trainable-weight sensitivity:",
      abs(k12 - local_similarity(Xq_train[0], Xq_train[1], w_probe + 0.4)))
print("Note: local overlap similarity is not automatically a Mercer/PSD kernel.")

## 8.1 — Quantum preflight (run before expensive training)

Checks the PennyLane circuit, data shape, finite similarities, and self-overlap. If this fails, stop before starting KTA.

In [ ]:
# Small quantum preflight before expensive KTA/Nyström loops
assert Xq_train.shape == (len(yq_train), N_QUBITS)
assert Xq_test.shape == (len(yq_test), N_QUBITS)
assert np.isfinite(Xq_train).all() and np.isfinite(Xq_test).all()
assert np.max(np.abs(Xq_train)) <= np.pi + 1e-12
assert np.max(np.abs(Xq_test)) <= np.pi + 1e-12
assert len(np.unique(yq_train)) == len(le.classes_)
assert len(np.unique(yq_test)) == len(le.classes_)

start = time.perf_counter()
preflight_K = np.array([
    [local_similarity(a, b, w_probe) for b in Xq_train[:3]]
    for a in Xq_train[:3]
])
elapsed = time.perf_counter() - start
assert np.isfinite(preflight_K).all()
assert ((preflight_K >= -1e-8) & (preflight_K <= 1 + 1e-8)).all()
assert np.allclose(np.diag(preflight_K), 1.0, atol=1e-6)
print("Quantum preflight PASSED.")
print("3×3 local similarity matrix:\n", np.round(preflight_K, 5))
print("Observed asymmetry (not necessarily a bug for this local readout):",
      float(np.max(np.abs(preflight_K - preflight_K.T))))
print("Preflight runtime:", round(elapsed, 3), "seconds")
print("Approximate raw kernel calls for training:",
      len(Xq_train) * N_LANDMARKS + len(Xq_test) * N_LANDMARKS + N_LANDMARKS**2)
print("This preflight checks the circuit and numerical outputs, not final model accuracy.")

## 9 — Train quantum kernel alignment (KTA)

Follow the corrected v3.1 implementation: COBYLA optimizes one rotation weight per qubit against a small **training-only** alignment subset. KTA must never see held-out test labels.

In [ ]:
align_rng = np.random.default_rng(SEED + 10_000)
align_idx = align_rng.choice(len(Xq_train), min(ALIGN_SUBSET, len(Xq_train)), replace=False)
X_align, y_align = Xq_train[align_idx], yq_train[align_idx]
Y_target = np.where(y_align[:, None] == y_align[None, :], 1.0, -1.0)
calls = [0]
def kta_loss(w):
    calls[0] += 1
    K = np.array([[local_similarity(a, b, w) for b in X_align] for a in X_align])
    denom = np.linalg.norm(K) * np.linalg.norm(Y_target) + 1e-9
    return -float(np.sum(K * Y_target) / denom)

w0 = 0.05 * np.random.default_rng(SEED).standard_normal(N_QUBITS)
t0 = time.perf_counter()
opt = minimize(kta_loss, w0, method="COBYLA",
               options={"maxiter": max(ALIGN_MAXITER, N_QUBITS + 2), "rhobeg": 0.3})
w_trained = np.asarray(opt.x, dtype=float)
align_seconds = time.perf_counter() - t0
print("KTA:", {"objective": -float(opt.fun), "calls": calls[0],
               "seconds": round(align_seconds, 1), "success": bool(opt.success)})
print("Trained weights:", np.round(w_trained, 4))

## 10 — Nyström approximation + train final linear SVM

Reuse the v3.1 Nyström construction. The local measurement matrix may be asymmetric or indefinite; we symmetrize its **landmark-landmark** block, report eigenvalues, and clip tiny/negative eigenvalues for a stable explicit feature map. This is an engineering approximation and does **not** establish that the original local similarity is a PSD quantum kernel.

In [ ]:
land_rng = np.random.default_rng(SEED + 20_000)
landmark_idx = land_rng.choice(len(Xq_train), min(N_LANDMARKS, len(Xq_train)), replace=False)
landmarks = Xq_train[landmark_idx].copy()

def cross_similarity(A, B, weights, log_every=0):
    out = np.empty((len(A), len(B)), dtype=float)
    for i, a in enumerate(A):
        for j, b in enumerate(B):
            out[i, j] = local_similarity(a, b, weights)
        if log_every and (i + 1) % log_every == 0:
            print(f"  {i+1}/{len(A)} rows")
    return out

t0 = time.perf_counter()
K_train_land = cross_similarity(Xq_train, landmarks, w_trained, log_every=100)
K_test_land = cross_similarity(Xq_test, landmarks, w_trained, log_every=60)
K_mm_raw = cross_similarity(landmarks, landmarks, w_trained)
K_mm = (K_mm_raw + K_mm_raw.T) / 2.0
eigvals, eigvecs = np.linalg.eigh(K_mm)
print("Landmark matrix: max asymmetry =", float(np.max(np.abs(K_mm_raw-K_mm_raw.T))))
print("Eigenvalue range:", float(eigvals.min()), float(eigvals.max()),
      "| negative eigenvalues:", int((eigvals < -1e-8).sum()))
eigvals_clipped = np.clip(eigvals, 1e-8, None)
K_mm_inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals_clipped)) @ eigvecs.T
psi_train = K_train_land @ K_mm_inv_sqrt
psi_test = K_test_land @ K_mm_inv_sqrt
qsvm = SVC(kernel="linear", C=SVM_C)
qsvm.fit(psi_train, yq_train)
pred_qsvm = qsvm.predict(psi_test)
kernel_seconds = time.perf_counter() - t0
print("QSVM finished in", round(kernel_seconds, 1), "seconds",
      "| feature shape:", psi_train.shape)

## 11 — Fair classical baselines on **identical rows**

RBF-SVM and Random Forest use the **same eight selected angle-scaled features**, with identical train/test indices. A second Random Forest uses all filtered StandardScaler features on the same rows as an informative additional reference.

In [ ]:
svm_rbf = SVC(kernel="rbf", C=SVM_C, gamma="scale")
rf_8 = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_all = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
svm_rbf.fit(Xq_train, yq_train)
rf_8.fit(Xq_train, yq_train)
rf_all.fit(Xtr_s[train_sel], yq_train)
pred_rbf = svm_rbf.predict(Xq_test)
pred_rf8 = rf_8.predict(Xq_test)
pred_rfall = rf_all.predict(Xte_s[test_sel])
predictions = {
    "QSVM v3.1 local": pred_qsvm,
    "RBF-SVM same 8": pred_rbf,
    "RF same 8": pred_rf8,
    "RF all filtered": pred_rfall,
}
rows = []
for name, pred in predictions.items():
    rows.append({"model": name, "accuracy": accuracy_score(yq_test, pred),
                 "macro_f1": f1_score(yq_test, pred, average="macro", zero_division=0),
                 "balanced_accuracy": balanced_accuracy_score(yq_test, pred),
                 "n_train": len(yq_train), "n_test": len(yq_test)})
metrics_df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
display(metrics_df)
metrics_df.to_csv(OUTPUT_DIR / "matched_model_metrics.csv", index=False)
print("NOTE: these are NEW real-data results; do not substitute historical CTGAN scores.")

## 12 — Evaluation figures and report

In [ ]:
print("QSVM classification report:")
print(classification_report(yq_test, pred_qsvm, target_names=list(le.classes_), zero_division=0))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(yq_test, pred_qsvm,
    display_labels=le.classes_, ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Corrected QSVM v3.1 — held-out real data")
axes[1].barh(metrics_df["model"], metrics_df["macro_f1"], color="#4466a8")
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Macro-F1")
axes[1].set_title("Matched subset comparison")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "evaluation.png", dpi=180, bbox_inches="tight")
plt.show()
evaluation = {
    "task": TASK, "seed": SEED, "dataset_sha256": sha,
    "n_train_subset": len(yq_train), "n_test_subset": len(yq_test),
    "n_train_pool": len(y_train), "n_test_pool": len(y_test),
    "classes": list(le.classes_), "selected_features": selected_feature_names,
    "device": DEVICE_NAME, "quantum_seconds": kernel_seconds,
    "kta_seconds": align_seconds,
    "metrics": rows,
    "warnings": ["No test-set tuning", "Local similarity may be indefinite",
                 "Quantum test subset is smaller than full real-data holdout"]
}
(OUTPUT_DIR / "evaluation_metrics.json").write_text(json.dumps(evaluation, indent=2))
print("Saved evaluation files to", OUTPUT_DIR)

## 13 — Save a complete, portable model bundle (.joblib)

A quantum QNode cannot reliably be serialized as a live circuit. Save the **circuit configuration**, fitted preprocessing objects, trained quantum weights, Nyström landmarks/transform and final SVM. The dashboard will reconstruct the corrected circuit using the same Python package versions.

**Security:** load joblib/pickle files only from trusted sources. Preserve the \`environment.json\` to reproduce the Python environment.

In [ ]:
bundle = {
    "format_version": "qtagger-v3.1-1",
    "task": TASK,
    "dataset_sha256": sha,
    "classes": list(le.classes_),
    "raw_feature_columns": raw_feature_cols,
    "fill_values": fill_values.to_dict(),
    "kept_columns": kept_columns,
    "removed_correlated_columns": drop_corr,
    "standard_scaler": standard_scaler,
    "selected_indices": selected_idx.astype(int),
    "selected_feature_names": selected_feature_names,
    "robust_scaler": robust_scaler,
    "quantum_config": {
        "n_qubits": N_QUBITS, "layers": LAYERS,
        "pairs": pairs, "device": DEVICE_NAME, "seed": SEED
    },
    "kta_weights": w_trained,
    "landmarks": landmarks,
    "K_mm_inv_sqrt": K_mm_inv_sqrt,
    "svm": qsvm,
    "evaluation": evaluation,
}
model_path = OUTPUT_DIR / "qtagger_v3_1.joblib"
joblib.dump(bundle, model_path, compress=3)
environment = {"python": platform.python_version(), "numpy": np.__version__,
               "pandas": pd.__version__, "scipy": scipy.__version__,
               "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
               "pennylane": qml.__version__, "joblib": joblib.__version__}
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
print("Saved:", model_path, "| size:", round(model_path.stat().st_size/1024, 1), "KiB")

## 14 — Reload and run inference from **raw** memory-feature rows

This function reproduces the complete training-time transformation in the correct order. It does not need labels, \`Filename\`, or \`Category\`. It requires the original numerical feature columns in their saved schema. Additional columns are ignored; missing required columns raise an error.

In [ ]:
def make_predictor(saved):
    qcfg = saved["quantum_config"]
    nq, depth = int(qcfg["n_qubits"]), int(qcfg["layers"])
    try:
        predict_dev = qml.device("lightning.qubit", wires=nq)
    except Exception:
        predict_dev = qml.device("default.qubit", wires=nq)
    masks = []
    ids = np.arange(2**nq)
    for a, b in qcfg["pairs"]:
        masks.append((((ids >> (nq-1-a)) & 1) == 0) &
                     (((ids >> (nq-1-b)) & 1) == 0))
    @qml.qnode(predict_dev)
    def probs(x1, x2, w):
        for _ in range(depth):
            qml.AngleEmbedding(x1, wires=range(nq), rotation="Y")
            for i in range(nq):
                qml.RY(w[i], wires=i)
            for i in range(nq):
                qml.CNOT(wires=[i, (i+1) % nq])
        for _ in range(depth):
            for i in reversed(range(nq)):
                qml.CNOT(wires=[i, (i+1) % nq])
            for i in range(nq):
                qml.RY(-w[i], wires=i)
            qml.adjoint(qml.AngleEmbedding)(x2, wires=range(nq), rotation="Y")
        return qml.probs(wires=range(nq))
    def predict(raw_df):
        missing = sorted(set(saved["raw_feature_columns"]) - set(raw_df.columns))
        if missing:
            raise ValueError(f"Missing {len(missing)} raw feature columns: {missing[:12]}")
        X = raw_df[saved["raw_feature_columns"]].copy()
        X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        X = X.fillna(pd.Series(saved["fill_values"]))
        X = X[saved["kept_columns"]]
        X = np.sign(X) * np.log1p(np.abs(X))
        z = saved["standard_scaler"].transform(X)
        z = z[:, saved["selected_indices"]]
        z = np.clip(saved["robust_scaler"].transform(z), -3, 3) / 3 * np.pi
        weights = saved["kta_weights"]
        landmarks = saved["landmarks"]
        K = np.array([[float(np.mean([np.asarray(probs(x, lp, weights))[m].sum()
                                       for m in masks])) for lp in landmarks] for x in z])
        psi = K @ saved["K_mm_inv_sqrt"]
        pred_idx = saved["svm"].predict(psi)
        return np.array(saved["classes"], dtype=object)[pred_idx]
    return predict

loaded = joblib.load(model_path)
predict_reloaded = make_predictor(loaded)
smoke_raw = X_test_raw.iloc[test_sel[:5]].copy()
pred_from_disk = predict_reloaded(smoke_raw)
expected = le.inverse_transform(pred_qsvm[:5])
assert np.array_equal(pred_from_disk, expected), (
    f"Reload mismatch: {pred_from_disk} vs {expected}"
)
print("Reload test passed — predictions match original QSVM.")
print(pd.DataFrame({"true": y_test_text[test_sel[:5]],
                    "predicted": pred_from_disk}))

## 15 — Export a demonstration input CSV and predictions

The dashboard can accept a CSV with the exact numerical feature columns. The demo file below contains **held-out real feature records** without target labels. A dashboard should not claim it scans raw executable files; QTagger+ classifies **already-extracted memory features**.

In [ ]:
demo_csv = OUTPUT_DIR / "demo_unlabeled_features.csv"
smoke_raw.to_csv(demo_csv, index=False)
pd.DataFrame({"predicted_family": pred_from_disk}).to_csv(
    OUTPUT_DIR / "demo_predictions.csv", index=False)
print("Demo CSV:", demo_csv)
print("Predictions:", OUTPUT_DIR / "demo_predictions.csv")
print("Final model:", model_path)
print("NEXT: use make_predictor(bundle) logic in qtagger/inference.py and connect Streamlit app.py.")

## Completion checklist and interpretation

1. **Data:** original CIC-MalMem-2022 CSV, SHA-256 recorded, numeric schema audited.
2. **Preprocessing:** original corrected v3 filter → signed log → StandardScaler → XGBoost top 8 → RobustScaler → angle map, with train-only fitting.
3. **Training:** corrected 8-qubit local QSVM v3.1, trainable KTA, Nyström and linear SVM.
4. **Comparison:** RBF-SVM and RF on identical train/test samples; no unsupported quantum-advantage claim.
5. **Persistence:** \`qtagger_v3_1.joblib\`, metrics JSON, environment JSON and demo CSV.
6. **Inference:** saved bundle reloads and reproduces the original five predictions exactly.

**Not included:** a trained model (you must run the notebook), CTGAN/QGAN retraining, executable-file scanning, production-ready API, or Streamlit dashboard. The dashboard is the next implementation stage after this notebook executes successfully.

**Reproducibility:** model performance may differ from historical GitHub reports because this notebook uses the **original real CSV**, removes exact duplicates, fixes train-only preprocessing, and evaluates on a new 120-row held-out subset. For a research-quality final result, also run more seeds and a larger held-out test set.